# Kibocsátási és gazdasági adatok összegyűjtése
### Barna Péter Bálint
Az alábbi notebook a tőzsdei "ticker" alapján betölti a cégeket, megszerzi az üzleti adataikat Yahoo Finance-ről a saját készítésű getEconomicData modullal, majd megszerzi az elérhető kibocsátási adatokat a saját készítésű getGHGData alapján.

In [ ]:
import pandas as pd
from getEconomicData import getEconomicData # Saját függvény, az üzleti adatok ('name','stock','gics_sector','gics_industry','market_cap') megszerzéséhez
import getGHGData

In [ ]:
def buildFinalDf():
    columns = ['name','stock','gics_sector','gics_industry','market_cap','disclosure_yr','total_ghg','scope1','scope2','scope2_lb','scope2_mb','scope3','co2e/usd','net0yr','light_data','links']
    df = pd.DataFrame(columns=columns)
    return df

In [ ]:
tickers = []
rText = ""
with open('tickersForAnalysis.txt', 'r') as f: # Inputként a keresendő cégek tőzsdei 'Ticker-jét kell megadni'
    rText = f.read()

tickers = rText.split('\n')

In [ ]:
companies_stocks = pd.DataFrame(tickers[:-1])

In [ ]:
economyData = getEconomicData(companies_stocks[0])

In [ ]:
economyData.drop_duplicates(subset=['name'], keep='first',inplace=True)
economyData.dropna(inplace=True)
economyData.to_csv("EconomicDataSaveHalfway.csv")
economyData

In [ ]:
for i in range(200,len(companies_stocks),100):
    ghgData = getGHGData.getGHGData(economyData['name'][i:i+100]) # Az NZDPU csak 100-asával engedte lekérni egy kérésben az adatokat.
    ghgDataFrame = pd.DataFrame(ghgData)
    ghgDataFrame.to_csv("GHGData2.csv", index=False)
    # Merge 2 dataframes on name
    merged_df = pd.merge(economyData, ghgDataFrame, on='name')
    merged_df.to_csv(f"{i}-{i+100}.csv")

In [ ]:
ghgDataFrame.to_csv("GHGData2.csv", index=False)

## Utólagos feldolgozás
### Leírás
* name: a vállalkozás teljes neve
* stock: részvény név
* gics_industry: Iparkategória a GICS szabvány alapján
* total_revenue: teljes árbevétel [$]
* total_ghg: teljes üvegházhatású gáz kibocsátva [CO2e]
* scope1
* scope2
* scope3: egyes scope-okban végzett kibocsátás
* co2e/usd: CO2-vel egyenértékű üvegházhatáskibocsátás/teljes árbevétel
* net0year: Tervezett net0 dátum, ha van /FELHASZNÁLÓ TÖLTI KI!!!/
* light_data: mérőszám, mely megadja, hogy mennyire volt könnyű megtalálni az adatot
    * 0 - NZDPU-n is fent van, foglalkoznak vele, könnyű
    * 1 - Net0trackeren fent van, egész könnyű
    * 2 - Létezik fenntarthatósági report, nehezebb
    * 3 - Nem talált fenntarthatósági reportot sem, gyakorlatilag nem foglalkoznak vele
* links: első 3 találat a fenntarthatósági reportokra

In [ ]:
# Merge 2 dataframes on name
merged_df = pd.merge(economyData, ghgDataFrame, on='name')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("400-rest.csv")